In [ ]:
# --- Cell 1: Setup (paths, imports, knobs) ---
from pathlib import Path
import time, pandas as pd

from project_package.modeling import (
    run_kmeans_from_csv,
    run_isolation_forest_from_csv,
    run_pca_embeddings_from_csv,
)

ROOT = Path.cwd()

CSV = ROOT / "ncr_ride_bookings_with_weather_filled_scaled_short.csv"
if not CSV.exists():
    alt = ROOT / "datasets" / "processed" / CSV.name
    if alt.exists():
        CSV = alt
assert CSV.exists(), f"CSV not found: {CSV}"

IDS = ["Booking ID", "Customer ID"]

# >>> write to ROOT (same level as Model), matching your supervised notebook
ART = ROOT / "unsupervised"
ART.mkdir(parents=True, exist_ok=True)

K_LIST = [3, 5, 8, 10]
RANDOM_STATE = 42

def mins_since(t0): 
    return (time.time() - t0) / 60.0


In [2]:
# --- Cell 2: KMeans clustering (full data) ---
t0 = time.time()
km = run_kmeans_from_csv(
    csv_path=str(CSV),
    id_cols=IDS,
    artifacts_dir=str(ART),
    k_list=K_LIST,
    random_state=RANDOM_STATE,
)
elapsed = mins_since(t0)

print("=== [KMEANS — FULL] ===")
print("Best k         :", km.best_k)
print("Metric report  :", km.report)                 # silhouette/DBI/CH per k
print("Labels CSV     :", km.labels_csv_path)
print("PCA(2D) CSV    :", km.pca2_csv_path)
print("Model (.pkl)   :", km.model_path)
print("Time (min)     :", round(elapsed, 2))

# Quick peek (optional, comment out if file is large)
try:
    print("\n[labels CSV head]")
    display(pd.read_csv(km.labels_csv_path, nrows=5))
    print("\n[pca2 CSV head]")
    display(pd.read_csv(km.pca2_csv_path, nrows=5))
except Exception as e:
    print("Preview skipped:", e)


=== [KMEANS — FULL] ===
Best k         : 5
Metric report  : {3: {'silhouette': 0.35617228164309783, 'dbi': 0.9040456859036338, 'ch': 104734.98890529951}, 5: {'silhouette': 0.35754847645730187, 'dbi': 0.9120198191103537, 'ch': 121282.36381564596}, 8: {'silhouette': 0.3175149628109835, 'dbi': 0.9002097176084035, 'ch': 119013.4512993234}, 10: {'silhouette': 0.3126682128051229, 'dbi': 0.9422701262130773, 'ch': 114871.87712162221}}
Labels CSV     : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised\kmeans_k5_labels.csv
PCA(2D) CSV    : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised\kmeans_k5_pca2.csv
Model (.pkl)   : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised\kmeans_k5.pkl
Time (min)     : 13.68

[labels CSV head]


,cluster,Booking ID,Customer ID
0,3,"""CNR5884300""","""CID1982111"""
1,3,"""CNR1326809""","""CID4604802"""
2,4,"""CNR8494506""","""CID9202816"""
3,1,"""CNR8906825""","""CID2610914"""
4,1,"""CNR1950162""","""CID9933542"""



[pca2 CSV head]


,pc1,pc2,cluster,Booking ID,Customer ID
0,-34.674380,0.516420,3,"""CNR5884300""","""CID1982111"""
1,-72.945809,1.061108,3,"""CNR1326809""","""CID4604802"""
2,7.533123,1.294543,4,"""CNR8494506""","""CID9202816"""
3,41.193525,58.275354,1,"""CNR8906825""","""CID2610914"""
4,36.233192,32.593871,1,"""CNR1950162""","""CID9933542"""


In [3]:
# --- Cell 3: Isolation Forest (full data) ---
t0 = time.time()
iso = run_isolation_forest_from_csv(
    csv_path=str(CSV),
    id_cols=IDS,
    artifacts_dir=str(ART),
    contamination=0.02,         # tweak if you expect more/fewer outliers
    random_state=RANDOM_STATE,
)
elapsed = mins_since(t0)

print("=== [ISOLATION FOREST — FULL] ===")
print("Contamination  :", iso.contamination)
print("Scores CSV     :", iso.scores_csv_path)        # has is_outlier + anomaly_score
print("Model (.pkl)   :", iso.model_path)
print("Time (min)     :", round(elapsed, 2))

try:
    print("\n[scores CSV head]")
    display(pd.read_csv(iso.scores_csv_path, nrows=5))
except Exception as e:
    print("Preview skipped:", e)


=== [ISOLATION FOREST — FULL] ===
Contamination  : 0.02
Scores CSV     : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised\isoforest_scores.csv
Model (.pkl)   : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised\isoforest.pkl
Time (min)     : 0.29

[scores CSV head]


,is_outlier,anomaly_score,Booking ID,Customer ID
0,0,-0.406494,"""CNR5884300""","""CID1982111"""
1,0,-0.424689,"""CNR1326809""","""CID4604802"""
2,0,-0.407539,"""CNR8494506""","""CID9202816"""
3,0,-0.445516,"""CNR8906825""","""CID2610914"""
4,0,-0.408780,"""CNR1950162""","""CID9933542"""


In [4]:
# --- Cell 4: PCA embeddings (full data) ---
t0 = time.time()
emb = run_pca_embeddings_from_csv(
    csv_path=str(CSV),
    id_cols=IDS,
    artifacts_dir=str(ART),
    n_components=2,
    random_state=RANDOM_STATE,
)
elapsed = mins_since(t0)

print("=== [PCA — FULL] ===")
print("Method         :", emb.method)
print("Components     :", emb.components)
print("Embed CSV      :", emb.embed_csv_path)         # pc1, pc2 (+ IDs)
print("Time (min)     :", round(elapsed, 2))

try:
    print("\n[embed CSV head]")
    display(pd.read_csv(emb.embed_csv_path, nrows=5))
except Exception as e:
    print("Preview skipped:", e)


=== [PCA — FULL] ===
Method         : pca
Components     : 2
Embed CSV      : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised\pca_2d.csv
Time (min)     : 0.08

[embed CSV head]


,pc1,pc2,Booking ID,Customer ID
0,-34.674380,0.516420,"""CNR5884300""","""CID1982111"""
1,-72.945809,1.061108,"""CNR1326809""","""CID4604802"""
2,7.533123,1.294543,"""CNR8494506""","""CID9202816"""
3,41.193525,58.275354,"""CNR8906825""","""CID2610914"""
4,36.233192,32.593871,"""CNR1950162""","""CID9933542"""


Lite below

In [ ]:
# --- Cell 1: Setup (paths, imports, knobs) ---
from pathlib import Path
import time, pandas as pd

from project_package.modeling import (
    load_csv_dedup, EXCLUDE_ALWAYS,
    run_kmeans_from_csv,
    run_isolation_forest_from_csv,
    run_pca_embeddings_from_csv,
)

ROOT = Path.cwd()

CSV = ROOT / "ncr_ride_bookings_with_weather_filled_scaled_short.csv"
if not CSV.exists():
    alt = ROOT / "datasets" / "processed" / CSV.name
    if alt.exists():
        CSV = alt
assert CSV.exists(), f"CSV not found: {CSV}"

IDS = ["Booking ID", "Customer ID"]

# >>> write to ROOT-level folders for lite artifacts
ART_LITE = ROOT / "unsupervised_lite"
ART_LITE.mkdir(parents=True, exist_ok=True)

SAMPLE_FRAC = 0.30
RANDOM_STATE = 42
K_LIST = [3, 5, 8, 10]

def mins_since(t0): 
    return (time.time() - t0) / 60.0


In [6]:
# --- Cell 2: Build a LITE CSV by sampling rows (stratification not required for unsupervised) ---
def make_unsup_lite_csv(csv_path: Path, sample_frac: float, seed: int) -> Path:
    """
    Create a smaller CSV for fast unsupervised runs:
      - Uses load_csv_dedup (drops duplicate columns incl. .1/.2 variants).
      - Randomly samples rows (no target => no stratification).
      - Writes next to the original file with a suffix.
    """
    df = load_csv_dedup(str(csv_path))
    if 0 < sample_frac < 1.0:
        df = df.sample(frac=sample_frac, random_state=seed).reset_index(drop=True)
    out = csv_path.with_name(csv_path.stem + f"__lite_s{int(sample_frac*100)}.csv")
    df.to_csv(out, index=False)
    return out

CSV_LITE = make_unsup_lite_csv(CSV, SAMPLE_FRAC, RANDOM_STATE)
print("LITE CSV:", CSV_LITE, "| exists:", CSV_LITE.exists())


LITE CSV: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\ncr_ride_bookings_with_weather_filled_scaled_short__lite_s30.csv | exists: True


In [7]:
# --- Cell 3: KMeans (LITE) ---
t0 = time.time()
km_lite = run_kmeans_from_csv(
    csv_path=str(CSV_LITE),
    id_cols=IDS,
    artifacts_dir=str(ART_LITE),
    k_list=K_LIST,
    random_state=RANDOM_STATE,
)
elapsed = mins_since(t0)

print("=== [KMEANS — LITE] ===")
print("Best k         :", km_lite.best_k)
print("Metric report  :", km_lite.report)
print("Labels CSV     :", km_lite.labels_csv_path)
print("PCA(2D) CSV    :", km_lite.pca2_csv_path)
print("Model (.pkl)   :", km_lite.model_path)
print("Time (min)     :", round(elapsed, 2))

try:
    print("\n[labels CSV head]")
    display(pd.read_csv(km_lite.labels_csv_path, nrows=5))
    print("\n[pca2 CSV head]")
    display(pd.read_csv(km_lite.pca2_csv_path, nrows=5))
except Exception as e:
    print("Preview skipped:", e)


=== [KMEANS — LITE] ===
Best k         : 5
Metric report  : {3: {'silhouette': 0.35763828900078554, 'dbi': 0.9073313937839166, 'ch': 31492.399355382833}, 5: {'silhouette': 0.35832935699306806, 'dbi': 0.9111420973927988, 'ch': 36504.620644632276}, 8: {'silhouette': 0.3179238526524699, 'dbi': 0.9037344457575678, 'ch': 35707.56134058397}, 10: {'silhouette': 0.31335372527981925, 'dbi': 0.9413394656217717, 'ch': 34481.86914032611}}
Labels CSV     : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised_lite\kmeans_k5_labels.csv
PCA(2D) CSV    : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised_lite\kmeans_k5_pca2.csv
Model (.pkl)   : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised_lite\kmeans_k5.pkl
Time (min)     : 1.33

[labels CSV head]


,cluster,Booking ID,Customer ID
0,4,"""CNR9907493""","""CID6858115"""
1,3,"""CNR1653982""","""CID4606212"""
2,1,"""CNR3151351""","""CID5996744"""
3,0,"""CNR5839286""","""CID6748555"""
4,3,"""CNR7329923""","""CID6785564"""



[pca2 CSV head]


,pc1,pc2,cluster,Booking ID,Customer ID
0,-8.723257,24.114367,4,"""CNR9907493""","""CID6858115"""
1,-92.955484,70.613415,3,"""CNR1653982""","""CID4606212"""
2,35.797753,37.174379,1,"""CNR3151351""","""CID5996744"""
3,-75.018852,-36.908231,0,"""CNR5839286""","""CID6748555"""
4,-62.935401,27.746881,3,"""CNR7329923""","""CID6785564"""


In [8]:
# --- Cell 4: Isolation Forest (LITE) ---
t0 = time.time()
iso_lite = run_isolation_forest_from_csv(
    csv_path=str(CSV_LITE),
    id_cols=IDS,
    artifacts_dir=str(ART_LITE),
    contamination=0.02,
    random_state=RANDOM_STATE,
)
elapsed = mins_since(t0)

print("=== [ISOLATION FOREST — LITE] ===")
print("Contamination  :", iso_lite.contamination)
print("Scores CSV     :", iso_lite.scores_csv_path)
print("Model (.pkl)   :", iso_lite.model_path)
print("Time (min)     :", round(elapsed, 2))

try:
    print("\n[scores CSV head]")
    display(pd.read_csv(iso_lite.scores_csv_path, nrows=5))
except Exception as e:
    print("Preview skipped:", e)


=== [ISOLATION FOREST — LITE] ===
Contamination  : 0.02
Scores CSV     : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised_lite\isoforest_scores.csv
Model (.pkl)   : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised_lite\isoforest.pkl
Time (min)     : 0.08

[scores CSV head]


,is_outlier,anomaly_score,Booking ID,Customer ID
0,0,-0.387165,"""CNR9907493""","""CID6858115"""
1,0,-0.468050,"""CNR1653982""","""CID4606212"""
2,0,-0.443805,"""CNR3151351""","""CID5996744"""
3,0,-0.408356,"""CNR5839286""","""CID6748555"""
4,0,-0.418447,"""CNR7329923""","""CID6785564"""


In [9]:
# --- Cell 5: PCA embeddings (LITE) ---
t0 = time.time()
emb_lite = run_pca_embeddings_from_csv(
    csv_path=str(CSV_LITE),
    id_cols=IDS,
    artifacts_dir=str(ART_LITE),
    n_components=2,
    random_state=RANDOM_STATE,
)
elapsed = mins_since(t0)

print("=== [PCA — LITE] ===")
print("Method         :", emb_lite.method)
print("Components     :", emb_lite.components)
print("Embed CSV      :", emb_lite.embed_csv_path)
print("Time (min)     :", round(elapsed, 2))

try:
    print("\n[embed CSV head]")
    display(pd.read_csv(emb_lite.embed_csv_path, nrows=5))
except Exception as e:
    print("Preview skipped:", e)
 

=== [PCA — LITE] ===
Method         : pca
Components     : 2
Embed CSV      : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised_lite\pca_2d.csv
Time (min)     : 0.02

[embed CSV head]


,pc1,pc2,Booking ID,Customer ID
0,-8.723257,24.114367,"""CNR9907493""","""CID6858115"""
1,-92.955484,70.613415,"""CNR1653982""","""CID4606212"""
2,35.797753,37.174379,"""CNR3151351""","""CID5996744"""
3,-75.018852,-36.908231,"""CNR5839286""","""CID6748555"""
4,-62.935401,27.746881,"""CNR7329923""","""CID6785564"""
